# Daily Weather Pipeline

This walkthrough loads the last **30 complete calendar dates ending yesterday** for Hyderabad, Bengaluru and Delhi. Each city uses its configured local timezone for weather aggregation; the window and Airflow schedule use UTC calendar dates consistently.

**Open-Meteo archive API -> raw.weather_daily -> staging.stg_weather -> marts.fct_city_daily**

Every pipeline stage below imports the same functions used by the Airflow DAG. SQL queries here only inspect results; they do not implement an alternative pipeline. Run `make up` followed by `make reproduce` for a fresh-kernel execution in Docker.

## 1. Configuration and Date Window

The window is calculated once. Exactly three cities times 30 dates means **90 rows** in the requested window. Dependencies are installed in the shared Airflow/Jupyter image; no local Python setup or API key is required.

In [1]:
from datetime import datetime, timedelta, timezone
from importlib.metadata import version
import json

import pandas as pd
from IPython.display import display

from ingestion.pipeline import PROJECT_ROOT, backfill, extract_date, load_cities, load_date, query_rows
from ingestion.transform import run_dbt

run_started_at = datetime.now(timezone.utc)
end_date = run_started_at.date() - timedelta(days=1)
start_date = end_date - timedelta(days=29)
cities = load_cities()
city_ids = [city['id'] for city in cities]
expected_rows = len(cities) * 30
window_parameters = (start_date, end_date, city_ids)

print(f'Execution started: {run_started_at.isoformat()}')
print(f'Inclusive window: {start_date} to {end_date}; expected rows: {expected_rows}')
print({package: version(package) for package in ('apache-airflow', 'dbt-core', 'dbt-postgres', 'pandas')})
display(pd.DataFrame(cities))

Execution started: 2026-09-18T04:08:22.956051+00:00
Inclusive window: 2026-08-19 to 2026-09-17; expected rows: 90
{'apache-airflow': '2.10.5', 'dbt-core': '1.8.8', 'dbt-postgres': '1.8.2', 'pandas': '2.2.3'}


,id,name,latitude,longitude,timezone
0,hyderabad,Hyderabad,17.3850,78.4867,Asia/Kolkata
1,bengaluru,Bengaluru,12.9716,77.5946,Asia/Kolkata
2,delhi,Delhi,28.6139,77.2090,Asia/Kolkata


## 2. Extract One Logical Date

`extract_date(end_date)` makes one archive request per city for the same explicit date. HTTP calls have a 5-second connect timeout, a 45-second read timeout, and up to three retries for connection failures, rate limiting and transient server errors.

The complete response is preserved as JSONB: field names and values are unchanged, while whitespace and object-key order are not retained. Date, unit and completeness checks reject invalid responses without substituting zeros or silently changing the date range. The raw metadata columns identify the configured city and requested date separately from the API's nearest-grid coordinates.

Recent archive data can be delayed or revised. If yesterday is incomplete, execution fails visibly so the same logical date can be retried later.

In [2]:
records = extract_date(end_date)
assert len(records) == len(cities)
print(f'Extracted {len(records)} responses for {end_date}')
display(pd.DataFrame([{key: row[key] for key in ('city_id', 'city_name', 'logical_date')} for row in records]))
print('Complete first API response, before loading:')
print(json.dumps(records[0]['payload'], indent=2, ensure_ascii=True))

Extracted 3 responses for 2026-09-17


,city_id,city_name,logical_date
0,hyderabad,Hyderabad,2026-09-17
1,bengaluru,Bengaluru,2026-09-17
2,delhi,Delhi,2026-09-17


Complete first API response, before loading:
{
  "latitude": 17.398945,
  "longitude": 78.457085,
  "generationtime_ms": 0.2843141555786133,
  "utc_offset_seconds": 19800,
  "timezone": "Asia/Kolkata",
  "timezone_abbreviation": "GMT+5:30",
  "elevation": 505.0,
  "daily_units": {
    "time": "iso8601",
    "temperature_2m_max": "\u00b0C",
    "temperature_2m_min": "\u00b0C",
    "temperature_2m_mean": "\u00b0C",
    "precipitation_sum": "mm",
    "wind_speed_10m_max": "km/h"
  },
  "daily": {
    "time": [
      "2026-09-17"
    ],
    "temperature_2m_max": [
      33.4
    ],
    "temperature_2m_min": [
      26.0
    ],
    "temperature_2m_mean": [
      29.9
    ],
    "precipitation_sum": [
      2.1
    ],
    "wind_speed_10m_max": [
      10.7
    ]
  }
}


## 3. Load and Prove Rerun Safety

`load_date` validates every city before opening a transaction. A primary key on `(city_id, logical_date)` and `INSERT ... ON CONFLICT DO UPDATE` make retries safe, including concurrent attempts to load the same key. All cities for one date commit together or roll back together.

The second call below uses exactly the same extracted responses. Counts, payloads and timestamps must remain unchanged. A later re-extraction may contain revised weather or changed API metadata; that replaces the existing payload rather than adding a row. JSONB retains the latest response, not a history of revisions.

In [3]:
first_load = load_date(end_date, records)
first_snapshot = query_rows(
    'select * from raw.weather_daily where logical_date = %s order by city_id', (end_date,)
)
second_load = load_date(end_date, records)
second_snapshot = query_rows(
    'select * from raw.weather_daily where logical_date = %s order by city_id', (end_date,)
)
assert first_load['rows_for_date'] == second_load['rows_for_date'] == len(cities)
assert first_snapshot == second_snapshot
expected_payloads = {row['city_id']: row['payload'] for row in records}
assert all(row['payload'] == expected_payloads[row['city_id']] for row in second_snapshot)

display(pd.DataFrame([{'attempt': 'first', **first_load}, {'attempt': 'second', **second_load}]))
print('Rerun proof: same row count, keys, payloads and timestamps. Raw payload equals extracted response.')
display(pd.DataFrame(second_snapshot))

,attempt,logical_date,rows_loaded,rows_for_date
0,first,2026-09-17,3,3
1,second,2026-09-17,3,3


Rerun proof: same row count, keys, payloads and timestamps. Raw payload equals extracted response.


,city_id,city_name,logical_date,payload,ingested_at
0,bengaluru,Bengaluru,2026-09-17,"{'daily': {'time': ['2026-09-17'], 'precipitat...",2026-09-18 04:08:24.227502+00:00
1,delhi,Delhi,2026-09-17,"{'daily': {'time': ['2026-09-17'], 'precipitat...",2026-09-18 04:08:24.227502+00:00
2,hyderabad,Hyderabad,2026-09-17,"{'daily': {'time': ['2026-09-17'], 'precipitat...",2026-09-18 04:08:24.227502+00:00


## 4. Backfill the Remaining Dates

`backfill(start_date, end_date - 1 day)` calls the same one-date extraction and loading functions. The latest date is already loaded above. Each date is its own transaction: a failure does not erase completed dates, and restarting the range is safe.

Coverage is checked against the expected city/date grid, not only the total row count. The warehouse may retain earlier history after future reproductions; assertions and reports deliberately filter the requested 30-day window.

In [4]:
backfill_results = backfill(start_date, end_date - timedelta(days=1))
assert len(backfill_results) == 29
print(f'Backfilled {len(backfill_results)} additional dates.')
display(pd.DataFrame(backfill_results).tail())

raw_keys = query_rows('''
    select city_id, logical_date from raw.weather_daily
    where logical_date between %s and %s and city_id = any(%s)
    order by city_id, logical_date
''', window_parameters)
expected_keys = {(city_id, start_date + timedelta(days=offset)) for city_id in city_ids for offset in range(30)}
assert len(raw_keys) == expected_rows
assert {(row['city_id'], row['logical_date']) for row in raw_keys} == expected_keys
raw_coverage = query_rows('''
    select city_name, count(*) as rows, count(distinct logical_date) as dates,
           min(logical_date) as first_date, max(logical_date) as last_date
    from raw.weather_daily
    where logical_date between %s and %s and city_id = any(%s)
    group by city_name order by city_name
''', window_parameters)
display(pd.DataFrame(raw_coverage))
print(f'Raw coverage verified: {len(raw_keys)} unique city/date rows.')

Backfilled 29 additional dates.


,logical_date,rows_loaded,rows_for_date
24,2026-09-12,3,3
25,2026-09-13,3,3
26,2026-09-14,3,3
27,2026-09-15,3,3
28,2026-09-16,3,3


,city_name,rows,dates,first_date,last_date
0,Bengaluru,30,30,2026-08-19,2026-09-17
1,Delhi,30,30,2026-08-19,2026-09-17
2,Hyderabad,30,30,2026-08-19,2026-09-17


Raw coverage verified: 90 unique city/date rows.


## 5. Transform with dbt

`run_dbt('run')` is the DAG's transformation callable. Staging extracts and casts the raw fields, trims city text and converts blank text to null. Bad values are exposed to tests rather than silently discarded. The mart retains the city/day grain and adds daily temperature range and `is_wet_day` (at least 1 mm of precipitation).

The API already provides daily aggregates, so the mart does not average averages or invent observation-level data. A staging view and full-refresh mart table keep reruns straightforward at this scale; rebuilding is inexpensive for this small dataset. Every dbt command has a ten-minute timeout and a nonzero exit fails the notebook or Airflow task.

In [5]:
run_dbt('run')
run_artifact = json.loads((PROJECT_ROOT / 'dbt/target/run_results.json').read_text())
display(pd.DataFrame([{'model': result['unique_id'], 'status': result['status'], 'seconds': result['execution_time']} for result in run_artifact['results']]))

stage_counts = query_rows('''
    select 'raw' as stage, count(*) as rows from raw.weather_daily
      where logical_date between %s and %s and city_id = any(%s)
    union all
    select 'staging', count(*) from staging.stg_weather
      where weather_date between %s and %s and city_id = any(%s)
    union all
    select 'mart', count(*) from marts.fct_city_daily
      where weather_date between %s and %s and city_id = any(%s)
''', window_parameters * 3)
assert all(row['rows'] == expected_rows for row in stage_counts)
display(pd.DataFrame(stage_counts))
display(pd.DataFrame(query_rows('''
    select * from staging.stg_weather
    where weather_date between %s and %s and city_id = any(%s)
    order by weather_date desc, city_name limit 6
''', window_parameters)))
display(pd.DataFrame(query_rows('''
    select * from marts.fct_city_daily
    where weather_date between %s and %s and city_id = any(%s)
    order by weather_date desc, city_name limit 6
''', window_parameters)))

04:08:58  Running with dbt=1.8.8
04:08:58  Registered adapter: postgres=1.8.2
04:08:59  Found 2 models, 45 data tests, 1 source, 427 macros
04:08:59  
04:08:59  Concurrency: 2 threads (target='dev')
04:08:59  
04:08:59  1 of 2 START sql view model staging.stg_weather ................................ [RUN]
04:08:59  1 of 2 OK created sql view model staging.stg_weather ........................... [CREATE VIEW in 0.16s]
04:08:59  2 of 2 START sql table model marts.fct_city_daily .............................. [RUN]
04:08:59  2 of 2 OK created sql table model marts.fct_city_daily ......................... [SELECT 90 in 0.13s]
04:08:59  
04:08:59  Finished running 1 view model, 1 table model in 0 hours 0 minutes and 0.60 seconds (0.60s).
04:09:00  
04:09:00  Completed successfully
04:09:00  
04:09:00  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 TOTAL=2



,model,status,seconds
0,model.weather_pipeline.stg_weather,success,0.160947
1,model.weather_pipeline.fct_city_daily,success,0.134336


,stage,rows
0,raw,90
1,staging,90
2,mart,90


,city_id,city_name,weather_date,timezone,temperature_min_c,temperature_max_c,temperature_mean_c,precipitation_mm,wind_speed_max_kmh,ingested_at
0,bengaluru,Bengaluru,2026-09-17,Asia/Kolkata,20.5,30.5,25.2,5.8,11.7,2026-09-18 04:08:24.227502+00:00
1,delhi,Delhi,2026-09-17,Asia/Kolkata,24.8,32.5,29.1,2.0,13.5,2026-09-18 04:08:24.227502+00:00
2,hyderabad,Hyderabad,2026-09-17,Asia/Kolkata,26.0,33.4,29.9,2.1,10.7,2026-09-18 04:08:24.227502+00:00
3,bengaluru,Bengaluru,2026-09-16,Asia/Kolkata,20.5,30.0,25.5,5.4,8.3,2026-09-18 04:08:57.280677+00:00
4,delhi,Delhi,2026-09-16,Asia/Kolkata,27.7,35.0,31.2,0.3,7.8,2026-09-18 04:08:57.280677+00:00
5,hyderabad,Hyderabad,2026-09-16,Asia/Kolkata,24.8,33.4,29.4,2.0,8.5,2026-09-18 04:08:57.280677+00:00


,city_id,city_name,weather_date,timezone,temperature_min_c,temperature_max_c,temperature_mean_c,temperature_range_c,precipitation_mm,is_wet_day,wind_speed_max_kmh,ingested_at
0,bengaluru,Bengaluru,2026-09-17,Asia/Kolkata,20.5,30.5,25.2,10.0,5.8,True,11.7,2026-09-18 04:08:24.227502+00:00
1,delhi,Delhi,2026-09-17,Asia/Kolkata,24.8,32.5,29.1,7.7,2.0,True,13.5,2026-09-18 04:08:24.227502+00:00
2,hyderabad,Hyderabad,2026-09-17,Asia/Kolkata,26.0,33.4,29.9,7.4,2.1,True,10.7,2026-09-18 04:08:24.227502+00:00
3,bengaluru,Bengaluru,2026-09-16,Asia/Kolkata,20.5,30.0,25.5,9.5,5.4,True,8.3,2026-09-18 04:08:57.280677+00:00
4,delhi,Delhi,2026-09-16,Asia/Kolkata,27.7,35.0,31.2,7.3,0.3,False,7.8,2026-09-18 04:08:57.280677+00:00
5,hyderabad,Hyderabad,2026-09-16,Asia/Kolkata,24.8,33.4,29.4,8.6,2.0,True,8.5,2026-09-18 04:08:57.280677+00:00


## 6. Data Quality and Documentation

The tests check composite keys, nulls, physical ranges, minimum/mean/maximum temperature ordering, raw date agreement and exact staging-to-mart values. The earlier coverage assertion also catches missing city/date combinations that uniqueness tests alone cannot catch.

These are deliberately broad sanity bounds, not a promise that the weather is accurate. Source, model and column descriptions are versioned in dbt YAML. `dbt docs generate` creates the catalog and lineage artifacts alongside the test results. Any failed dbt test raises an error and stops this walkthrough.

In [6]:
run_dbt('test')
test_artifact = json.loads((PROJECT_ROOT / 'dbt/target/run_results.json').read_text())
test_results = pd.DataFrame([
    {'test': result['unique_id'], 'status': result['status'], 'failures': result.get('failures')}
    for result in test_artifact['results']
])
assert len(test_results) > 0 and test_results['status'].eq('pass').all()
display(test_results.groupby('status', as_index=False).size().rename(columns={'size': 'tests'}))
run_dbt('docs')
print('Documentation generated:', (PROJECT_ROOT / 'dbt/target/index.html').is_file())
assert (PROJECT_ROOT / 'dbt/target/catalog.json').is_file()

04:09:03  Running with dbt=1.8.8
04:09:04  Registered adapter: postgres=1.8.2
04:09:04  Found 2 models, 45 data tests, 1 source, 427 macros
04:09:04  
04:09:05  Concurrency: 2 threads (target='dev')
04:09:05  
04:09:05  1 of 45 START test between_values_fct_city_daily_precipitation_mm__2000__0 ..... [RUN]
04:09:05  2 of 45 START test between_values_fct_city_daily_temperature_max_c__60___90 .... [RUN]
04:09:05  2 of 45 PASS between_values_fct_city_daily_temperature_max_c__60___90 .......... [PASS in 0.13s]
04:09:05  1 of 45 PASS between_values_fct_city_daily_precipitation_mm__2000__0 ........... [PASS in 0.14s]
04:09:05  3 of 45 START test between_values_fct_city_daily_temperature_mean_c__60___90 ... [RUN]
04:09:05  4 of 45 START test between_values_fct_city_daily_temperature_min_c__60___90 .... [RUN]
04:09:05  4 of 45 PASS between_values_fct_city_daily_temperature_min_c__60___90 .......... [PASS in 0.09s]
04:09:05  3 of 45 PASS between_values_fct_city_daily_temperature_mean_c__60___90 

,status,tests
0,pass,45


04:09:11  Running with dbt=1.8.8
04:09:11  Registered adapter: postgres=1.8.2
04:09:12  Found 2 models, 45 data tests, 1 source, 427 macros
04:09:12  
04:09:12  Concurrency: 2 threads (target='dev')
04:09:12  
04:09:13  Building catalog
04:09:13  Catalog written to /opt/airflow/dbt/target/catalog.json

Documentation generated: True


## 7. Business View: Recent Weather by City

An operations planner can compare the last 30 days by average temperature, hottest day, total rain, number of wet days and strongest wind. Each city must contribute all 30 dates before the comparison is accepted. Wet-day counts use the documented 1 mm reporting threshold and are not a forecast or an operational safety recommendation.

In [7]:
city_report = pd.DataFrame(query_rows('''
    select city_name as city, count(*) as days,
           round(avg(temperature_mean_c), 1) as average_temperature_c,
           max(temperature_max_c) as hottest_temperature_c,
           sum(precipitation_mm) as total_rainfall_mm,
           count(*) filter (where is_wet_day) as wet_days,
           max(wind_speed_max_kmh) as strongest_wind_kmh
    from marts.fct_city_daily
    where weather_date between %s and %s and city_id = any(%s)
    group by city_id, city_name order by total_rainfall_mm desc, city_name
''', window_parameters))
assert len(city_report) == len(cities) and city_report['days'].eq(30).all()
display(city_report)
wettest_city = city_report.iloc[0]
print(f"From {start_date} to {end_date}, {wettest_city['city']} had the most rain among these cities: "
      f"{wettest_city['total_rainfall_mm']} mm, with {wettest_city['wet_days']} wet days.")

,city,days,average_temperature_c,hottest_temperature_c,total_rainfall_mm,wet_days,strongest_wind_kmh
0,Delhi,30,29.2,35.0,154.6,19,14.4
1,Bengaluru,30,24.4,31.0,114.1,25,22.3
2,Hyderabad,30,27.0,33.4,73.9,18,21.5


From 2026-08-19 to 2026-09-17, Delhi had the most rain among these cities: 154.6 mm, with 19 wet days.


## 8. Airflow Uses the Same Code

The DAG is `extract -> load -> dbt_run -> dbt_test`, scheduled daily in UTC. Both extraction and loading receive `{{ ds }}`, not the wall-clock date. A scheduled interval for September 17 runs after that interval closes on September 18 and requests September 17 weather for each city's local day. Backfill supplies the same logical date explicitly.

The DAG starts paused and uses `catchup=False` so a clean review does not launch unrequested API calls or overlap the notebook. Unpause it to enable daily scheduling. Explicit `airflow dags backfill` still works. `max_active_runs=1` serializes DAG runs; avoid running the notebook while a scheduled run is rebuilding the shared mart. Each task has two retries with exponential backoff and a 15-minute timeout.

Only three small responses pass through XCom per date. Larger datasets would need external object storage, incremental models and stronger coordination, which are unnecessary for this assessment.

Example after `make up`:

```bash
docker compose exec -T airflow airflow dags backfill weather_daily --start-date 2026-09-17 --end-date 2026-09-17 --yes
```

The check below inspects the real DAG and proves its callables are the imports used above; it does not pretend that DAG inspection executes a scheduled run.

In [8]:
from airflow.models import DagBag

bag = DagBag(dag_folder=str(PROJECT_ROOT / 'dags'), include_examples=False)
assert not bag.import_errors, bag.import_errors
weather_dag = bag.dags['weather_daily']
assert weather_dag.get_task('extract').python_callable is extract_date
assert weather_dag.get_task('load').python_callable is load_date
assert weather_dag.get_task('dbt_run').python_callable is run_dbt
assert weather_dag.get_task('dbt_test').python_callable is run_dbt
assert weather_dag.get_task('extract').op_kwargs['logical_date'] == '{{ ds }}'
assert weather_dag.get_task('load').op_kwargs['logical_date'] == '{{ ds }}'

display(pd.DataFrame([
    {'task': task.task_id, 'callable': task.python_callable.__name__,
     'next_tasks': ', '.join(sorted(task.downstream_task_ids)),
     'retries': task.retries, 'timeout_minutes': task.execution_timeout.total_seconds() / 60}
    for task in weather_dag.topological_sort()
]))
print(f'DAG: {weather_dag.dag_id}; schedule: {weather_dag.schedule_interval}; timezone: {weather_dag.timezone}')
print(f'Verified {expected_rows} city/day rows and {len(test_results)} passing dbt tests.')

[2026-09-18T04:09:18.482+0000] {dagbag.py:588} INFO - Filling up the DagBag from /opt/airflow/dags


,task,callable,next_tasks,retries,timeout_minutes
0,extract,extract_date,load,2,15.0
1,load,load_date,dbt_run,2,15.0
2,dbt_run,run_dbt,dbt_test,2,15.0
3,dbt_test,run_dbt,,2,15.0


DAG: weather_daily; schedule: @daily; timezone: UTC
Verified 90 city/day rows and 45 passing dbt tests.
